In [ ]:
!pip install -q sentence-transformers pymupdf transformers torch pytesseract pdf2image pillow

In [ ]:
import fitz
import numpy as np
import torch
from PIL import Image
import pytesseract
from pdf2image import convert_from_path

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

In [ ]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
def extract_text_from_pdf(file_path):
    text = ""

    # Try normal extraction first
    doc = fitz.open(file_path)
    for page in doc:
        text += page.get_text()

    # If text is too small → assume handwritten → use OCR
    if len(text.strip()) < 50:
        print("🔍 Using OCR for handwritten detection...")
        images = convert_from_path(file_path)

        for img in images:
            text += pytesseract.image_to_string(img)

    return text

In [ ]:
def chunk_text(text, size=400):
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]

In [ ]:
human_samples = [
    "I tried solving the problem but made mistakes initially.",
    "At first I was confused but later I understood the concept.",
    "We performed the experiment and recorded values manually.",
    "I forgot a bracket which caused an error in the program.",
    "Sometimes I had to retry multiple times to get correct output.",
    "This assignment reflects my own understanding.",
    "I learned step by step and improved gradually.",
    "There were errors but I fixed them after debugging.",
    "I discussed problems with my classmates to understand better.",
    "It was not easy but I managed to complete it."
]

ai_samples = [
    "This comprehensive analysis explores the subject in a structured manner.",
    "The findings demonstrate a highly optimized and systematic approach.",
    "It is important to consider multiple dimensions in a holistic framework.",
    "The results indicate significant improvements with enhanced efficiency.",
    "This study provides a detailed and coherent understanding of the topic.",
    "The methodology ensures scalability and precision.",
    "The framework facilitates optimized performance and structured output.",
    "The approach highlights clarity, consistency, and analytical depth.",
    "This model integrates multiple perspectives with high accuracy.",
    "The results demonstrate robust and scalable implementation."
]

human_emb = embed_model.encode(human_samples)
ai_emb = embed_model.encode(ai_samples)

In [ ]:
def calculate_perplexity(text):
    encodings = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = gpt2(**encodings, labels=encodings["input_ids"])
    return torch.exp(outputs.loss).item()

In [ ]:
def normalize(val, min_val, max_val):
    return max(0, min(1, (val - min_val) / (max_val - min_val + 1e-5)))

In [ ]:
def detect(text_chunks):
    results = []

    for chunk in text_chunks:
        emb = embed_model.encode([chunk])

        # Semantic
        ai_sim = cosine_similarity(emb, ai_emb).mean()
        human_sim = cosine_similarity(emb, human_emb).mean()
        semantic_score = ai_sim / (ai_sim + human_sim + 1e-5)

        # Perplexity
        ppl = calculate_perplexity(chunk[:400])
        ppl_score = 1 - normalize(ppl, 20, 80)

        # Burstiness
        sentence_lengths = [len(s.split()) for s in chunk.split('.') if len(s.split()) > 0]
        burst = np.std(sentence_lengths) if len(sentence_lengths) > 1 else 0
        burst_score = 1 - normalize(burst, 0, 25)

        # FINAL SCORE (STRONG WEIGHT)
        final_score = (
            0.85 * semantic_score +
            0.1 * ppl_score +
            0.05 * burst_score
        )

        # Amplify difference
        final_score = final_score ** 0.7

        final_score = max(0, min(1, final_score))

        results.append({
            "text": chunk,
            "ai_prob": round(final_score * 100, 2)
        })

    return results

In [ ]:
def analyze(file_path):
    text = extract_text_from_pdf(file_path)

    if len(text.strip()) == 0:
        print("❌ No text detected. Try better scan.")
        return

    chunks = chunk_text(text)
    results = detect(chunks)

    avg = np.mean([r["ai_prob"] for r in results])

    print("\n===== FINAL RESULT =====\n")
    print(f"AI Probability: {round(avg,2)}%\n")

    if avg < 35:
        verdict = "Likely Human-written"
    elif avg < 65:
        verdict = "Mixed / Uncertain"
    else:
        verdict = "Likely AI-generated"

    print("Final Verdict:", verdict)

    print("\n--- Suspicious Sections ---\n")

    for i, r in enumerate(results):
        if r["ai_prob"] > 60:
            print(f"Chunk {i+1}: {r['ai_prob']}% AI-like")
            print(r["text"][:200], "\n")

In [ ]:
from google.colab import files
uploaded = files.upload()

file_path = list(uploaded.keys())[0]
analyze(file_path)

Saving human_low_ai_assignment.pdf to human_low_ai_assignment.pdf

===== FINAL RESULT =====

AI Probability: 53.74%

Final Verdict: Mixed / Uncertain

--- Suspicious Sections ---

